# Puntos 9 y 12 – Balance hídrico de largo plazo y evaluación de los productos
**Trabajo 1 de Hidroclimatología · Cuenca del río Cauca hasta Hidroituango**

Para periodos largos, $\Delta S \approx 0$, y el balance queda

$$P - ETR \approx R, \qquad Q_{estimado} = (P - ETR)\,A$$

- **Punto 9:** se calcula $Q_{estimado}$ con la precipitación del IDEAM en el área de drenaje de cada estación
  hidrológica (punto 8), se compara con el caudal observado y se discuten las fuentes de incertidumbre.
- **Punto 12 – Evaluación de los productos mediante el balance hídrico:** se repite el balance usando como
  precipitación de entrada cada producto (IDEAM, CHIRPS, ERA5-Land y ERA5), se construye la tabla del enunciado
  (P anual, ETR, Q estimado, Q observado y error) y se determina qué producto representa mejor las condiciones
  hidroclimáticas de la cuenca. La evaluación no se limita a comparar la precipitación: se mide la capacidad de
  cada producto para reproducir el balance hídrico observado.

La ETR de cada producto se calcula con Budyko **celda a celda**, usando la misma ETP de Hargreaves (punto 6) y la P
de ese producto.

**Archivos necesarios:**
- `ETP_anual.tif` (punto 6)
- `P_IDEAM_anual.tif` (punto 3) y `P_CHIRPS_anual.tif`, `P_ERA5L_anual.tif`, `P_ERA5_anual.tif` (punto 11)
- `areas_drenaje_estaciones.gpkg` y `caudales_estaciones.csv` (punto 8)
- `cuenca_cauca.gpkg`

## 0. Instalación de librerías
Solo si no las instaló antes. Después de instalar, reinicie el kernel.

In [ ]:
%pip install numpy pandas matplotlib geopandas rasterio pyproj shapely

## 1. Librerías, parámetros y funciones de apoyo

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from scipy.optimize import brentq

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)

# ----------------------------- PARÁMETROS -----------------------------
CRS = "EPSG:3116"
CUENCA = "cuenca_cauca.gpkg"
RIO = "rio_principal.gpkg"
ETP_ANUAL = "ETP_anual.tif"
AREAS = "areas_drenaje_estaciones.gpkg"
CAUDALES = "caudales_estaciones.csv"
COBERTURA_MINIMA = 0.95       # fracción mínima del área de drenaje que debe caer dentro de los mapas
ESTACION_CIERRE = "26237050"  # punto 12: Puente Pescadero, la estación más cercana a la presa (≈ 6 km aguas arriba).
                              # Si no quedó apta en el punto 8, se usa la estación válida de mayor área de drenaje.
TOLERANCIA = 20               # punto 12: error (%) que se considera aceptable en el caudal estimado
# ----------------------------------------------------------------------

PRODUCTOS = ["IDEAM", "CHIRPS", "ERA5L", "ERA5"]
NOMBRE = {"CHIRPS": "CHIRPS", "ERA5L": "ERA5-Land", "ERA5": "ERA5", "IDEAM": "IDEAM"}
COLOR = {"IDEAM": "#252525", "CHIRPS": "#1b9e77", "ERA5L": "#d95f02", "ERA5": "#7570b3"}
SEG_ANIO = 365.25 * 86400

In [ ]:
# ----------------------- Funciones de apoyo SIG -----------------------
import rasterio
from rasterio import features
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from pyproj import Transformer
from shapely.geometry import shape
from shapely.ops import unary_union


def leer_raster(ruta):
    # Devuelve (arreglo enmascarado, info). Si el archivo tiene varias bandas: arreglo (bandas, filas, columnas)
    with rasterio.open(ruta) as src:
        arr = src.read(masked=True).astype("float64")
        info = {"transform": src.transform, "crs": src.crs, "shape": (src.height, src.width),
                "profile": src.profile.copy()}
    return (arr[0] if arr.shape[0] == 1 else arr), info


def guardar_raster(ruta, arr, info, nodata=-9999.0, descripciones=None, dtype="float32"):
    a = np.ma.masked_invalid(np.ma.asarray(arr, dtype="float64"))
    if a.ndim == 2:
        a = a[None]
    perfil = info["profile"].copy()
    perfil.update(driver="GTiff", dtype=dtype, nodata=nodata, count=a.shape[0], compress="lzw")
    with rasterio.open(ruta, "w", **perfil) as dst:
        dst.write(a.filled(nodata).astype(dtype))
        for i, d in enumerate(descripciones or [], start=1):
            dst.set_band_description(i, d)


def coords_celdas(info):
    # Coordenadas (x, y) del centro de cada celda, en el SRC del ráster
    T = info["transform"]
    filas, cols = np.indices(info["shape"])
    return T.a * (cols + 0.5) + T.b * (filas + 0.5) + T.c, T.d * (cols + 0.5) + T.e * (filas + 0.5) + T.f


def a_lonlat(x, y, crs):
    lon, lat = Transformer.from_crs(crs, "EPSG:4326", always_xy=True).transform(np.asarray(x), np.asarray(y))
    return np.asarray(lon), np.asarray(lat)


def de_lonlat(lon, lat, crs):
    x, y = Transformer.from_crs("EPSG:4326", crs, always_xy=True).transform(np.asarray(lon), np.asarray(lat))
    return np.asarray(x), np.asarray(y)


def remuestrear_a_grilla(arr, lon, lat, info, metodo="bilinear"):
    # Lleva un campo en grilla regular lon/lat (2D o 3D con el tiempo primero) a la grilla del DEM
    arr = np.asarray(arr, dtype="float64")
    if lat[0] < lat[-1]:                       # filas de norte a sur
        arr, lat = arr[..., ::-1, :], lat[::-1]
    dx, dy = abs(lon[1] - lon[0]), abs(lat[1] - lat[0])
    src_T = from_origin(lon.min() - dx / 2, lat.max() + dy / 2, dx, dy)
    capas = arr if arr.ndim == 3 else arr[None]
    salida = np.full((capas.shape[0],) + info["shape"], np.nan)
    for k in range(capas.shape[0]):
        reproject(source=np.ascontiguousarray(capas[k]), destination=salida[k], src_transform=src_T,
                  src_crs="EPSG:4326", src_nodata=np.nan, dst_transform=info["transform"], dst_crs=info["crs"],
                  dst_nodata=np.nan, resampling=getattr(Resampling, metodo))
    return salida if arr.ndim == 3 else salida[0]


def mascara_geometria(geom, info):
    # True en las celdas cuyo centro cae dentro de la geometría
    return features.geometry_mask([geom], out_shape=info["shape"], transform=info["transform"], invert=True)


def poligonizar(mascara, info):
    geoms = [shape(g) for g, v in features.shapes(mascara.astype("uint8"), mask=mascara,
                                                    transform=info["transform"]) if v == 1]
    return unary_union(geoms)

In [ ]:
def decorar(ax, titulo, ext, cuenca, rio=None):
    # Límite de la cuenca, río, flecha de norte, barra de escala y SRC
    cuenca.boundary.plot(ax=ax, color="k", lw=0.8)
    if rio is not None:
        rio.plot(ax=ax, color="#08306b", lw=1.0)
    ax.annotate("", xy=(0.93, 0.93), xytext=(0.93, 0.84), xycoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>", color="k", lw=1.5))
    ax.text(0.93, 0.945, "N", transform=ax.transAxes, ha="center", fontsize=12, fontweight="bold")
    x0 = ext[0] + 0.06 * (ext[1] - ext[0]); y0 = ext[2] + 0.04 * (ext[3] - ext[2])
    ax.plot([x0, x0 + 50000], [y0, y0], "k-", lw=3)
    ax.text(x0 + 25000, y0 + 0.012 * (ext[3] - ext[2]), "50 km", ha="center", fontsize=8)
    ax.set_title(titulo, fontsize=11)
    ax.text(0.02, 0.005, "MAGNA-SIRGAS / Colombia Bogotá zone (EPSG:3116)", transform=ax.transAxes, fontsize=7,
            bbox=dict(facecolor="white", edgecolor="none", alpha=0.7, pad=1))
    ax.ticklabel_format(style="plain"); ax.tick_params(labelsize=7)
    ax.locator_params(axis="x", nbins=4)
    ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); ax.set_aspect("equal")


def extension(info):
    T = info["transform"]; h, w = info["shape"]
    return [T.c, T.c + T.a * w, T.f + T.e * h, T.f]


def mapa(arr, info, titulo, etiqueta, cuenca, rio=None, cmap="viridis", archivo=None, vmin=None, vmax=None, ax=None):
    ext = extension(info)
    propio = ax is None
    if propio:
        fig, ax = plt.subplots(figsize=(6, 9))
    im = ax.imshow(arr, extent=ext, cmap=cmap, vmin=vmin, vmax=vmax, interpolation="nearest")
    if propio:
        plt.colorbar(im, ax=ax, shrink=0.6, label=etiqueta)
    decorar(ax, titulo, ext, cuenca, rio)
    if propio:
        fig.tight_layout()
        if archivo:
            fig.savefig(archivo, dpi=200)
        plt.show()
    return im

cuenca = gpd.read_file(CUENCA).to_crs(CRS)
rio = gpd.read_file(RIO).to_crs(CRS) if os.path.exists(RIO) else None

## 2. ETR y escorrentía de cada producto (Budyko celda a celda)

In [ ]:
def budyko(P, ETP):
    phi = ETP / P
    return P * np.sqrt(phi * np.tanh(1 / phi) * (1 - np.exp(-phi))), phi


ETP, info = leer_raster(ETP_ANUAL)
celda_km2 = abs(info["transform"].a * info["transform"].e) / 1e6
P, ETR, R = {}, {}, {}
for prod in PRODUCTOS:
    ruta = f"P_{prod}_anual.tif"
    if not os.path.exists(ruta):
        print(f"{prod}: falta {ruta}; se omite"); continue
    p, _ = leer_raster(ruta)
    m = np.ma.getmaskarray(p) | np.ma.getmaskarray(ETP)
    P[prod] = np.ma.masked_array(p, mask=m)
    ETR[prod], _ = budyko(P[prod], np.ma.masked_array(ETP, mask=m))
    R[prod] = P[prod] - ETR[prod]
    guardar_raster(f"ETR_{prod}.tif", ETR[prod], info)
    guardar_raster(f"R_{prod}.tif", R[prod], info)
PRODUCTOS = list(P)
print("Productos disponibles:", PRODUCTOS)

fig, axs = plt.subplots(1, len(PRODUCTOS), figsize=(4.2 * len(PRODUCTOS), 8))
vmax = max(np.percentile(R[p].compressed(), 99) for p in PRODUCTOS)
for ax, prod in zip(np.atleast_1d(axs), PRODUCTOS):
    im = mapa(R[prod], info, f"R = P − ETR · {NOMBRE[prod]}\nmedia {R[prod].mean():,.0f} mm/año", "", cuenca, rio,
              cmap="Blues", vmin=0, vmax=vmax, ax=ax)
fig.colorbar(im, ax=axs, shrink=0.6, label="Escorrentía de largo plazo (mm/año)")
fig.savefig("fig_mapas_R_productos.png", dpi=200, bbox_inches="tight")
plt.show()

## 3. Balance en el área de drenaje de cada estación
Para cada área de drenaje se promedian P, ETP y ETR de las celdas que caen dentro, y se calcula:
- $R_{est} = \overline{P} - \overline{ETR}$, en mm/año;
- $Q_{est} = R_{est}\,A$, en m³/s;
- $Error = 100\,(Q_{est} - Q_{obs})/Q_{obs}$.

También se agrega la cuenca completa hasta Hidroituango. Allí no hay estación justo en el cierre, así que solo
se estima su caudal.

In [ ]:
sub = gpd.read_file(AREAS).to_crs(CRS)
qobs = pd.read_csv(CAUDALES, dtype={"codigo": str})
sub["codigo"] = sub["codigo"].astype(str)
sub = sub.merge(qobs[["codigo", "nombre", "Q_medio_m3s", "R_obs_mm", "inicio", "fin", "Q_usado"]],
                on="codigo", how="left", suffixes=("_gpkg", ""))
sub["nombre"] = sub["nombre"].fillna(sub.get("nombre_gpkg"))
sub["area_km2"] = sub["area_km2"].astype(float)

mascaras = {}
for _, s in sub.iterrows():
    m = mascara_geometria(s.geometry, info) & ~np.ma.getmaskarray(ETP)
    mascaras[s.codigo] = (s.nombre, s.area_km2, m, s.Q_medio_m3s)
mascaras["CUENCA"] = ("Cuenca hasta Hidroituango", (~np.ma.getmaskarray(ETP)).sum() * celda_km2,
                      ~np.ma.getmaskarray(ETP), np.nan)

filas = []
for cod, (nom, A, m, q) in mascaras.items():
    cobertura = m.sum() * celda_km2 / A
    for prod in PRODUCTOS:
        mp = m & ~np.ma.getmaskarray(P[prod])
        p_, etp_, etr_ = P[prod].data[mp].mean(), ETP.data[mp].mean(), ETR[prod].data[mp].mean()
        r_ = p_ - etr_
        q_est = r_ / 1000 * A * 1e6 / SEG_ANIO
        r_obs = q * SEG_ANIO / (A * 1e6) * 1000 if np.isfinite(q) else np.nan
        filas.append({"codigo": cod, "estacion": nom, "area_km2": A, "cobertura_%": 100 * cobertura,
                      "producto": NOMBRE[prod], "P_mm": p_, "ETP_mm": etp_, "phi": etp_ / p_, "ETR_mm": etr_,
                      "Q_est_mm": r_, "Q_obs_mm": r_obs, "Q_est_m3s": q_est, "Q_obs_m3s": q,
                      "Error_%": 100 * (q_est - q) / q if np.isfinite(q) else np.nan})
bal = pd.DataFrame(filas)
bal.to_csv("tabla_balance_productos.csv", index=False, encoding="utf-8-sig")
baja = bal[(bal["cobertura_%"] < 100 * COBERTURA_MINIMA) & (bal.codigo != "CUENCA")].estacion.unique()
if len(baja):
    print("Ojo: área de drenaje cubierta parcialmente por los mapas (estación fuera de la cuenca):", list(baja),
          "\n→ Esas estaciones no se usan en la evaluación.")

# Estaciones que entran en la evaluación: con caudal observado y área de drenaje completa dentro de los mapas
validas = bal[(bal.codigo != "CUENCA") & bal.Q_obs_m3s.notna() & (bal["cobertura_%"] >= 100 * COBERTURA_MINIMA)].copy()
print(f"Estaciones para la evaluación: {validas.estacion.nunique()}")

## 4. Punto 9 – Balance con la precipitación del IDEAM

In [ ]:
t9 = bal[bal.producto == "IDEAM"].drop(columns=["producto"]).set_index("estacion")
if t9.empty:
    print("No está P_IDEAM_anual.tif: cópielo del punto 3 a esta carpeta para el punto 9.")
display(t9[["area_km2", "cobertura_%", "P_mm", "ETP_mm", "phi", "ETR_mm", "Q_est_mm", "Q_obs_mm",
            "Q_est_m3s", "Q_obs_m3s", "Error_%"]].round(1))

### 4.1 Fuentes de incertidumbre
Primero se cuantifican dos cosas para la estación con mayor área de drenaje:
- **Sensibilidad:** cuánto cambia $Q_{est}$ si P o ETP cambian 10 %.
- **P necesaria:** qué factor habría que aplicar a la P del IDEAM para que el balance cierre exactamente.

In [ ]:
REF = "IDEAM" if "IDEAM" in P else PRODUCTOS[0]
if len(validas):
    principal = validas.sort_values("area_km2").iloc[-1]
    cod = principal.codigo
    nom, A, m, q = mascaras[cod]
    m = m & ~np.ma.getmaskarray(P[REF])
    p0, e0 = P[REF].data[m], ETP.data[m]

    def q_est(fp=1.0, fe=1.0):
        etr, _ = budyko(p0 * fp, e0 * fe)
        return (p0 * fp - etr).mean() / 1000 * A * 1e6 / SEG_ANIO

    base = q_est()
    sens = pd.DataFrame({
        "cambio": ["P +10 %", "P −10 %", "ETP +10 %", "ETP −10 %"],
        "Q estimado (m³/s)": [q_est(1.1), q_est(0.9), q_est(1, 1.1), q_est(1, 0.9)]})
    sens["cambio en Q (%)"] = 100 * (sens["Q estimado (m³/s)"] / base - 1)
    print(f"Estación: {nom} | Q estimado con {NOMBRE[REF]} = {base:,.0f} m³/s | Q observado = {q:,.0f} m³/s")
    display(sens.round(1))
    print(f"Elasticidad de Q a la precipitación ≈ {(q_est(1.01) / base - 1) / 0.01:.2f} "
          "(un error de 1 % en P produce ese % de error en Q)")
    try:
        k = brentq(lambda k: q_est(k) - q, 0.2, 5)
        print(f"Para cerrar el balance, la P de {NOMBRE[REF]} tendría que ser {100 * (k - 1):+.1f} % "
              f"({p0.mean() * k:,.0f} mm/año en lugar de {p0.mean():,.0f} mm/año)")
    except ValueError:
        print("No se encontró un factor de P entre 0,2 y 5 que cierre el balance.")
    info_q = qobs.set_index("codigo").loc[cod]
    print(f"Periodo de caudales de {nom}: {info_q.inicio} a {info_q.fin} (se usó: {info_q.Q_usado}); "
          "normales de P del IDEAM: 1981–2010")

#### Guía para la discusión de incertidumbre (ajústela a sus números)
- **Precipitación.**
  - Hay pocas estaciones en las zonas altas, donde suele llover más (efecto orográfico), así que la P de cuenca
    tiende a quedar **subestimada**.
  - Los pluviómetros además subestiman la lluvia con viento (captación).
  - Como la elasticidad de Q a P es mayor que 1, un error pequeño en P se amplifica en Q.
- **Evapotranspiración.**
  - Hargreaves-Samani usa solo temperatura y radiación extraterrestre, no humedad ni viento.
  - La curva de Budyko es genérica: no tiene parámetros de vegetación o suelo, como el ω de Fu.
  - Un ±10 % en ETP cambia Q en el porcentaje que muestra la tabla de sensibilidad.
- **Caudal.**
  - Los caudales salen de curvas de calibración nivel–caudal, con errores típicos de 5–15 %, mayores en crecientes.
  - Hay vacíos en las series.
  - El alto Cauca está regulado por Salvajina (desde 1985) y hay extracciones para riego y acueductos en el valle
    geográfico.
- **Interpolación espacial.**
  - IDW y kriging no representan el efecto orográfico (sección 7 del punto 3).
  - La validación cruzada del punto 3 da el orden de magnitud del error local.
  - Al promediar sobre áreas grandes parte de ese error se cancela.
- **Periodos.** Las normales de P son 1981–2010 y los caudales se promediaron en los años completos de 1981–2010.
  Si a una estación le faltan años muy húmedos (La Niña 1988–89, 1999–2000) o muy secos (El Niño 1991–92,
  1997–98), su caudal medio queda sesgado frente a la P.
- **ΔS ≈ 0 y aguas subterráneas.** Se asume que no hay cambio de almacenamiento ni flujo subterráneo que cruce la
  divisoria. Los embalses y acuíferos del valle pueden violar ligeramente este supuesto.

## 5. Punto 12 – Evaluación de los productos mediante el balance hídrico
Se repite el balance de largo plazo usando como precipitación de entrada **cada uno de los productos**: IDEAM,
CHIRPS, ERA5-Land y ERA5. Para cada uno:

$$ETR = P\,\sqrt{\phi\,\tanh(1/\phi)\,(1-e^{-\phi})},\quad \phi = \frac{ETP}{P}, \qquad
Q_{estimado} = P - ETR, \qquad Error = 100\,\frac{Q_{estimado}-Q_{observado}}{Q_{observado}}$$

La ETP de Hargreaves (punto 6) es la misma para todos; solo cambia la P.

La evaluación tiene cinco partes:
1. **Tabla del enunciado en el cierre de la cuenca** (sección 5.1).
2. **La misma tabla en cada estación** (5.2): un producto puede acertar en el cierre por compensación de errores.
3. **Error del caudal estimado** en todas las estaciones (5.3).
4. **Más allá de la precipitación** (5.4): qué P haría falta para cerrar el balance, si el balance es físicamente
   posible con la lluvia de cada producto y si el producto reproduce cómo cambia la escorrentía de una estación a otra.
5. **Síntesis** (5.5): qué producto o productos representan mejor la cuenca.

### 5.1 Tabla del enunciado en el cierre de la cuenca
No hay estación exactamente en la presa. Se usa **Puente Pescadero** (≈ 6 km aguas arriba), cuya área de drenaje es
prácticamente la cuenca completa; P, ETR y Q se promedian sobre esa área de drenaje.

In [ ]:
ORDEN = [NOMBRE[p] for p in PRODUCTOS]
COLOR_N = {NOMBRE[p]: COLOR[p] for p in PRODUCTOS}
INV = {v: k for k, v in NOMBRE.items()}
if validas.empty:
    raise RuntimeError("No hay estaciones con caudal observado y área de drenaje completa dentro de los mapas: revise el punto 8.")
if str(ESTACION_CIERRE) in set(validas.codigo):
    cierre = str(ESTACION_CIERRE)
else:
    cierre = validas.sort_values("area_km2").codigo.iloc[-1]
    print(f"La estación {ESTACION_CIERRE} no está entre las estaciones válidas (no quedó apta en el punto 8 o su área "
          "no está completa); se usa la estación válida de mayor área de drenaje.")
tc = validas[validas.codigo == cierre].set_index("producto").reindex(ORDEN)
NOM_CIERRE, A_CIERRE = tc.estacion.iloc[0], tc.area_km2.iloc[0]
A_CUENCA = bal.loc[bal.codigo == "CUENCA", "area_km2"].iloc[0]
print(f"Cierre de la evaluación: {NOM_CIERRE} ({cierre}) | área de drenaje {A_CIERRE:,.0f} km² "
      f"= {100 * A_CIERRE / A_CUENCA:.0f} % de la cuenca hasta Hidroituango")
print(f"Q observado = {tc.Q_obs_m3s.iloc[0]:,.0f} m³/s = {tc.Q_obs_mm.iloc[0]:,.0f} mm/año\n")

COLS12 = {"P_mm": "P anual (mm/año)", "ETR_mm": "ETR (mm/año)", "Q_est_mm": "Q estimado (mm/año)",
          "Q_obs_mm": "Q observado (mm/año)", "Error_%": "Error (%)"}
tabla12 = tc[list(COLS12)].rename(columns=COLS12).rename_axis("Producto")
display(tabla12.round(1))
tabla12.round(1).to_csv("tabla_punto12_cierre.csv", encoding="utf-8-sig")

fig, axs = plt.subplots(1, 2, figsize=(12.5, 4.8))
x = np.arange(len(ORDEN))
axs[0].bar(x - 0.2, tc.P_mm, 0.38, color="#4a90c2", label="P")
axs[0].bar(x + 0.2, tc.ETR_mm, 0.38, color="#e3a33b", label="ETR (Budyko)")
axs[0].set(xticks=x, xticklabels=ORDEN, ylabel="mm/año", title=f"P y ETR en el área de drenaje de {NOM_CIERRE}")
axs[0].legend(fontsize=9); axs[0].grid(axis="y", alpha=0.3)
qo = tc.Q_obs_mm.iloc[0]
axs[1].axhspan(qo * (1 - TOLERANCIA / 100), qo * (1 + TOLERANCIA / 100), color="gray", alpha=0.15,
               label=f"Q observado ± {TOLERANCIA} %")
axs[1].bar(x, tc.Q_est_mm, 0.55, color=[COLOR_N[p] for p in ORDEN], edgecolor="k", lw=0.4)
axs[1].axhline(qo, color="k", lw=1.4, ls="--", label=f"Q observado = {qo:,.0f} mm/año")
for xi, (v, e) in enumerate(zip(tc.Q_est_mm, tc["Error_%"])):
    axs[1].annotate(f"{e:+.1f} %", (xi, v), xytext=(0, 4), textcoords="offset points", ha="center", fontsize=9,
                    bbox=dict(facecolor="white", edgecolor="none", pad=1.5, alpha=0.85))
axs[1].set(xticks=x, xticklabels=ORDEN, ylabel="mm/año", ylim=(0, max(tc.Q_est_mm.max(), qo) * 1.25),
           title="Q estimado = P − ETR frente al Q observado")
axs[1].legend(fontsize=8, loc="upper left"); axs[1].grid(axis="y", alpha=0.3)
fig.tight_layout(); fig.savefig("fig_tabla12_cierre.png", dpi=200)
plt.show()

### 5.2 La misma tabla en cada estación

In [ ]:
info_est = validas.drop_duplicates("estacion").set_index("estacion")[["area_km2", "Q_obs_m3s", "Q_obs_mm"]]
piv_err = validas.pivot_table(index="estacion", columns="producto", values="Error_%").reindex(columns=ORDEN)
t_est = info_est.join(piv_err).sort_values("area_km2")
print("Error del caudal estimado (%) en cada estación, de aguas arriba a aguas abajo:")
display(t_est.round(1))
for est_, g in validas.sort_values("area_km2").groupby("estacion", sort=False):
    print(f"\n{est_}  (A = {g.area_km2.iloc[0]:,.0f} km²)")
    display(g.set_index("producto").reindex(ORDEN)[list(COLS12)].rename(columns=COLS12).round(1))

### 5.3 Error del caudal estimado en todas las estaciones

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
piv = validas.sort_values("area_km2").pivot_table(index="estacion", columns="producto", values="Error_%", sort=False)
piv = piv[[p for p in ORDEN if p in piv]]
piv.plot.bar(ax=axs[0], color=[COLOR_N[p] for p in piv.columns], width=0.8)
axs[0].axhline(0, color="k", lw=0.8)
for lim in (-TOLERANCIA, TOLERANCIA):
    axs[0].axhline(lim, color="gray", ls=":", lw=0.8)
axs[0].set(ylabel="Error en Q estimado (%)", xlabel="", title="Error del balance por estación y producto")
axs[0].tick_params(axis="x", rotation=30); axs[0].grid(axis="y", alpha=0.3)

for prod in ORDEN:
    g = validas[validas.producto == prod]
    axs[1].scatter(g.Q_obs_m3s, g.Q_est_m3s, s=45, color=COLOR_N[prod], edgecolor="k", lw=0.4, label=prod)
q = np.array([validas[["Q_obs_m3s", "Q_est_m3s"]].min().min() * 0.7, validas[["Q_obs_m3s", "Q_est_m3s"]].max().max() * 1.4])
axs[1].plot(q, q, "k-", lw=0.8, label="1:1")
axs[1].fill_between(q, q * (1 - TOLERANCIA / 100), q * (1 + TOLERANCIA / 100), color="gray", alpha=0.15,
                    label=f"± {TOLERANCIA} %")
axs[1].set(xscale="log", yscale="log", xlabel="Q observado (m³/s)", ylabel="Q estimado (m³/s)",
           title="Caudal estimado por balance vs. observado")
axs[1].grid(alpha=0.3, which="both"); axs[1].legend(fontsize=8)
fig.tight_layout(); fig.savefig("fig_balance_productos.png", dpi=200)
plt.show()

### 5.4 Más allá de la precipitación
Que un producto llueva parecido al IDEAM no garantiza que cierre el balance: el IDEAM también tiene error, sobre todo
en las zonas altas. Por eso se evalúa directamente **qué tan compatible es la lluvia de cada producto con el caudal
medido**:

- **P necesaria y corrección de P.** Se busca el factor $k$ tal que, multiplicando la P del producto por $k$ celda a
  celda, el balance da exactamente el caudal observado. La corrección $100\,(k-1)$ dice cuánta lluvia le falta
  (+) o le sobra (−) al producto.
- **Balance físicamente posible.** Con el caudal observado, la ETR "observada" es $ETR_{obs} = P - Q_{obs}$. Debe
  cumplir $0 \le ETR_{obs} \le ETP$:
  - $ETR_{obs} < 0$: el río lleva más agua de la que llueve; la P del producto es demasiado baja.
  - $ETR_{obs} > ETP$: se evaporaría más de lo que la energía disponible permite; la P es demasiado alta (o el caudal
    está disminuido por extracciones).
- **Reproducción espacial.** Correlación entre la escorrentía estimada y la observada (mm/año) en las estaciones:
  mide si el producto reproduce cómo cambia el rendimiento hídrico a lo largo del río, y no solo el total.
  Las áreas de drenaje están anidadas, así que este r se interpreta con cautela.

In [ ]:
def factor_P(cod, prod):
    # Factor k que, aplicado a la P del producto, hace que el balance dé exactamente el caudal observado
    nom, A, m, q = mascaras[cod]
    mp = m & ~np.ma.getmaskarray(P[prod])
    p0, e0 = P[prod].data[mp].astype(float), ETP.data[mp].astype(float)
    r_obs = q * SEG_ANIO / (A * 1e6) * 1000
    f = lambda k: (k * p0 - budyko(k * p0, e0)[0]).mean() - r_obs
    try:
        return brentq(f, 0.2, 5)
    except ValueError:
        return np.nan


validas["k_P"] = [factor_P(c, INV[p]) for c, p in zip(validas.codigo, validas.producto)]
validas["P_necesaria_mm"] = validas.k_P * validas.P_mm
validas["correccion_P_%"] = 100 * (validas.k_P - 1)
validas["ETR_obs_mm"] = validas.P_mm - validas.Q_obs_mm
validas["balance_posible"] = (validas.ETR_obs_mm >= 0) & (validas.ETR_obs_mm <= validas.ETP_mm)

c5 = validas[validas.codigo == cierre].set_index("producto").reindex(ORDEN)
t_cons = pd.DataFrame({"P del producto (mm/año)": c5.P_mm, "P necesaria (mm/año)": c5.P_necesaria_mm,
                       "Corrección de P (%)": c5["correccion_P_%"], "ETR obs = P − Q obs (mm/año)": c5.ETR_obs_mm,
                       "ETR Budyko (mm/año)": c5.ETR_mm, "ETP (mm/año)": c5.ETP_mm,
                       "¿Balance posible?": np.where(c5.balance_posible, "sí", "NO")}).rename_axis("Producto")
print(f"En el cierre ({NOM_CIERRE}):")
display(t_cons.round(1))
print("\nCorrección de P necesaria para cerrar el balance (%) en cada estación:")
display(validas.sort_values("area_km2").pivot_table(index="estacion", columns="producto", values="correccion_P_%",
                                                    sort=False).reindex(columns=ORDEN).round(1))

# Guardar la tabla de balance con las columnas nuevas (la usa el sitio web)
extra = validas[["codigo", "producto", "k_P", "P_necesaria_mm", "correccion_P_%", "ETR_obs_mm", "balance_posible"]]
bal = bal.drop(columns=[c for c in extra.columns if c not in ("codigo", "producto") and c in bal], errors="ignore")
bal = bal.merge(extra, on=["codigo", "producto"], how="left")
bal.to_csv("tabla_balance_productos.csv", index=False, encoding="utf-8-sig")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(13, 5.5))
for prod in ORDEN:
    g = validas[validas.producto == prod]
    gc = g[g.codigo == cierre]
    axs[0].scatter(g.P_necesaria_mm, g.P_mm, s=45, color=COLOR_N[prod], edgecolor="k", lw=0.4, label=prod)
    axs[0].scatter(gc.P_necesaria_mm, gc.P_mm, s=170, marker="*", color=COLOR_N[prod], edgecolor="k", lw=0.6)
    axs[1].scatter(g.Q_obs_mm, g.Q_est_mm, s=45, color=COLOR_N[prod], edgecolor="k", lw=0.4, label=prod)
    axs[1].scatter(gc.Q_obs_mm, gc.Q_est_mm, s=170, marker="*", color=COLOR_N[prod], edgecolor="k", lw=0.6)
for ax, cols, tol, tit, xl, yl in [
        (axs[0], ["P_necesaria_mm", "P_mm"], 10, "P del producto vs. P necesaria para cerrar el balance",
         "P necesaria (mm/año)", "P del producto (mm/año)"),
        (axs[1], ["Q_obs_mm", "Q_est_mm"], TOLERANCIA, "Escorrentía estimada vs. observada",
         "Q observado (mm/año)", "Q estimado = P − ETR (mm/año)")]:
    v = validas[cols].to_numpy(dtype=float)
    v = v[np.isfinite(v)]
    lim = np.array([v.min() * 0.85, v.max() * 1.1])
    ax.plot(lim, lim, "k-", lw=0.8, label="1:1")
    ax.fill_between(lim, lim * (1 - tol / 100), lim * (1 + tol / 100), color="gray", alpha=0.15, label=f"± {tol} %")
    ax.set(xlim=lim, ylim=lim, xlabel=xl, ylabel=yl, title=tit)
    ax.grid(alpha=0.3); ax.set_aspect("equal")
axs[0].legend(fontsize=8, loc="upper left")
axs[1].text(0.98, 0.02, f"★ = {NOM_CIERRE} (cierre)", transform=axs[1].transAxes, ha="right", fontsize=8)
fig.tight_layout(); fig.savefig("fig_evaluacion_productos.png", dpi=200)
plt.show()

**Las estaciones en el espacio de Budyko.** Con el caudal observado, cada área de drenaje queda en el punto
$(\phi,\ ETR_{obs}/P)$. Si cae **sobre** la curva, el producto cierra bien el balance:
- **Por encima** de la curva: el caudal observado es menor de lo que la curva predice. La P del producto puede estar
  sobrestimada, o el caudal disminuido.
- **Por debajo**: el caudal observado es mayor. La P puede estar subestimada, que es frecuente en zonas altas con
  pocas estaciones.
- **Por encima de 1 o por debajo de 0** en el eje vertical, o a la izquierda de la línea ETR = ETP: el balance no
  puede cerrar con esa lluvia.

In [ ]:
f = np.linspace(0.05, 2.5, 300)
fig, ax = plt.subplots(figsize=(7, 5.5))
ax.plot(f, np.sqrt(f * np.tanh(1 / f) * (1 - np.exp(-f))), "k-", lw=2, label="Curva de Budyko")
ax.plot([0, 1], [0, 1], "k--", lw=0.8, label="Límite de energía (ETR = ETP)")
ax.plot([1, 2.5], [1, 1], "k:", lw=0.8, label="Límite de agua (ETR = P)")
for prod in ORDEN:
    g = validas[validas.producto == prod]
    ax.scatter(g.phi, 1 - g.Q_obs_mm / g.P_mm, s=55, color=COLOR_N[prod], edgecolor="k", lw=0.4, label=prod)
ax.set(xlabel="φ = ETP/P", ylabel="ETR_obs / P = 1 − Q_obs/P", xlim=(0, max(1.2, validas.phi.max() * 1.1)),
       ylim=(min(0, (1 - validas.Q_obs_mm / validas.P_mm).min() - 0.05), 1.1), title="Estaciones en el espacio de Budyko")
ax.grid(alpha=0.3); ax.legend(fontsize=8)
fig.tight_layout(); fig.savefig("fig_budyko_estaciones.png", dpi=200)
plt.show()

### 5.5 Síntesis: ¿qué producto representa mejor la cuenca?
El criterio principal es la **capacidad de reproducir el caudal observado**:

$$\text{Puntaje del balance} = \tfrac{1}{2}\left(|Error_{cierre}| + \overline{|Error|}_{estaciones}\right)$$

Combina el total de la cuenca (tabla del enunciado) con lo que pasa a lo largo del río. Un producto puede acertar en
el cierre por compensación de errores y fallar en las estaciones de aguas arriba. Menor puntaje = mejor.

Las demás columnas explican el resultado:
- corrección de P necesaria;
- estaciones dentro de la tolerancia;
- r espacial;
- estaciones donde el balance es físicamente imposible.

La diferencia de P con el IDEAM se muestra solo como referencia y **no entra en el puntaje**. El IDEAM aparece en la
tabla como punto de comparación: la pregunta es si algún producto representa la cuenca tan bien o mejor que la
interpolación de las estaciones.

In [ ]:
p_ideam = tc.P_mm.get("IDEAM", np.nan)
DENTRO = f"Estaciones con |error| ≤ {TOLERANCIA} %"
filas = []
for prod in ORDEN:
    g = validas[validas.producto == prod]
    err_c, mae = tc.loc[prod, "Error_%"], g["Error_%"].abs().mean()
    filas.append({
        "Producto": prod,
        "P en el cierre (mm/año)": tc.loc[prod, "P_mm"],
        "Dif. de P con IDEAM (%)": 100 * (tc.loc[prod, "P_mm"] / p_ideam - 1) if np.isfinite(p_ideam) else np.nan,
        "Error de Q en el cierre (%)": err_c,
        "Corrección de P en el cierre (%)": c5.loc[prod, "correccion_P_%"],
        "Error medio en estaciones (%)": g["Error_%"].mean(),
        "Error absoluto medio (%)": mae,
        DENTRO: int((g["Error_%"].abs() <= TOLERANCIA).sum()),
        "Estaciones evaluadas": len(g),
        "r espacial (Q est. vs Q obs., mm)": g[["Q_est_mm", "Q_obs_mm"]].corr().iloc[0, 1] if len(g) >= 3 else np.nan,
        "Estaciones con balance imposible": int((~g.balance_posible).sum()),
        "Puntaje del balance (%)": (abs(err_c) + mae) / 2})
ev = pd.DataFrame(filas).set_index("Producto").sort_values("Puntaje del balance (%)")
ev.insert(0, "Puesto", np.arange(1, len(ev) + 1))
display(ev.round(2))
ev.round(3).to_csv("evaluacion_productos_balance.csv", encoding="utf-8-sig")
pd.Series({"codigo": cierre, "estacion": NOM_CIERRE, "area_km2": round(A_CIERRE, 1),
           "fraccion_cuenca_%": round(100 * A_CIERRE / A_CUENCA, 1), "Q_obs_m3s": round(tc.Q_obs_m3s.iloc[0], 1),
           "tolerancia_%": TOLERANCIA}).to_json("punto12_cierre.json", force_ascii=False)

# Resumen corto que usa el punto 14
resumen = pd.DataFrame({"error_medio_pct": ev["Error medio en estaciones (%)"],
                        "error_abs_medio_pct": ev["Error absoluto medio (%)"],
                        "error_cierre_pct": ev["Error de Q en el cierre (%)"],
                        "n_estaciones": ev["Estaciones evaluadas"]}).rename_axis("producto")
resumen.to_csv("resumen_balance_productos.csv", encoding="utf-8-sig")

In [ ]:
# Conclusión automática: revísela y complétela con su interpretación
EC, MAE, PJ = "Error de Q en el cierre (%)", "Error absoluto medio (%)", "Puntaje del balance (%)"
prods = ev.drop(index="IDEAM", errors="ignore")
mejor = prods.index[0] if len(prods) else ev.index[0]
e = ev.loc[mejor]
n_est = int(e["Estaciones evaluadas"])
txt = [f"Entre los productos, el que mejor reproduce el balance hídrico observado es {mejor}: error de {e[EC]:+.1f} % "
       f"en el caudal de {NOM_CIERRE} y error absoluto medio de {e[MAE]:.1f} % en {n_est} estaciones "
       f"(puntaje {e[PJ]:.1f} %)."]
if "IDEAM" in ev.index and mejor != "IDEAM":
    i = ev.loc["IDEAM"]
    comp = "mejor" if e[PJ] < i[PJ] else "peor"
    txt.append(f"Con la lluvia interpolada del IDEAM el puntaje es {i[PJ]:.1f} % (error de {i[EC]:+.1f} % en el cierre): "
               f"{mejor} cierra el balance {comp} que las estaciones del IDEAM.")
ok = ev.index[(ev[EC].abs() <= TOLERANCIA) & (ev[MAE] <= TOLERANCIA)]
ok_p = [p for p in ok if p != "IDEAM"]
txt.append(f"Productos con error ≤ {TOLERANCIA} % en el cierre y en promedio: "
           + (", ".join(ok_p) if ok_p else "ninguno") + ".")
signo = np.sign(tc["Error_%"].dropna())
if len(signo) and (signo < 0).all():
    txt.append("Todas las fuentes de lluvia subestiman el caudal en el cierre. Además de la precipitación, puede haber "
               "ETR sobrestimada (Hargreaves + Budyko) o lluvia en zonas altas que ninguna fuente capta.")
elif len(signo) and (signo > 0).all():
    txt.append("Todas las fuentes de lluvia sobrestiman el caudal en el cierre. Puede haber ETR subestimada, o "
               "extracciones y pérdidas que el balance no considera.")
else:
    txt.append("Unas fuentes sobrestiman y otras subestiman el caudal en el cierre: la diferencia la explica la "
               "precipitación de cada una.")
if len(prods) > 1 and prods["Dif. de P con IDEAM (%)"].notna().any():
    cercano = prods["Dif. de P con IDEAM (%)"].abs().idxmin()
    if cercano != mejor:
        txt.append(f"Comparar solo la precipitación no basta: el producto con la P más parecida a la del IDEAM es {cercano}, "
                   f"pero el que mejor reproduce el caudal observado es {mejor}.")
    else:
        txt.append(f"El producto con la P más parecida a la del IDEAM ({cercano}) es también el que mejor reproduce el "
                   "caudal observado.")
g = validas[validas.producto == mejor]
if (g["Error_%"] > 0).any() and (g["Error_%"] < 0).any():
    txt.append(f"El error de {mejor} cambia de signo entre estaciones (de {g['Error_%'].min():+.0f} % a "
               f"{g['Error_%'].max():+.0f} %): parte del acierto en el cierre puede ser compensación de errores.")
corr = e["Corrección de P en el cierre (%)"]
if np.isfinite(corr):
    txt.append(f"Para cerrar exactamente el balance en {NOM_CIERRE}, la P de {mejor} tendría que cambiar {corr:+.1f} %.")
for prod in ev.index[ev["Estaciones con balance imposible"] > 0]:
    n = int(ev.loc[prod, "Estaciones con balance imposible"])
    txt.append(f"{prod}: en {n} {'estación' if n == 1 else 'estaciones'} P − Q observado queda fuera de [0, ETP]; con esa "
               "lluvia el balance no puede cerrar.")
print("\n".join("• " + t for t in txt))
with open("conclusion_punto12.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(txt))

### 5.6 Guía para la redacción del punto 12 (ajústela a sus números)
- **Respuesta directa.** Diga qué producto o productos representan mejor la cuenca y con qué criterio: la tabla del
  enunciado (5.1), el error en todas las estaciones (5.3) y la síntesis (5.5).
- **No se queda en la precipitación.** Muestre que un producto puede tener una P parecida a la del IDEAM y aun así
  no cerrar el balance, o al revés. Use la corrección de P (5.4): un producto que necesita poca corrección es
  consistente con el caudal medido, aunque difiera del IDEAM.
- **Error sistemático o compensación.** Si un producto acierta en el cierre pero falla en las estaciones de aguas
  arriba, el acierto puede ser compensación de errores.
- **Relación con los puntos 10, 11 y 14.** Por ejemplo:
  - Los reanálisis (ERA5, ERA5-Land) suelen sobrestimar la lluvia sobre los Andes.
  - CHIRPS se corrige con estaciones y suele quedar más cerca de los pluviómetros.
  - Con resolución gruesa (ERA5, ≈ 28 km) se pierde el contraste entre el valle seco y las laderas húmedas.
- **Limitaciones.** El mismo modelo de ETR (Hargreaves + Budyko) se usa con todos los productos, así que un error de
  la ETR afecta a todos por igual. Los caudales también tienen error (5–15 %) y el alto Cauca está regulado.